# Lab 5: Kernels, Splines and Gaussian Processes

**ECON5129 Statistical Machine Learning** &middot; Adam Smith Business School, University of Glasgow

{{COLAB_BADGE}}

This lab accompanies Lecture 5. By the end of the session you should be able to:

1. Demonstrate the curse of dimensionality and explain why local methods stop being local.
2. Implement the Nadaraya-Watson kernel smoother and the local linear estimator, and diagnose boundary bias.
3. Fit regression splines and a penalised spline, choosing smoothness rather than knot positions.
4. Write any of these as a linear smoother $\widehat{\mathbf{y}} = \mathbf{S}\mathbf{y}$ and measure its effective degrees of freedom as $\operatorname{tr}(\mathbf{S})$.
5. Apply the kernel trick through kernel ridge regression, working in an infinite-dimensional basis without ever forming it.
6. Fit a Gaussian process, sample from its prior and posterior, and read the role of its hyperparameters.

**Plan for the session**

| Time | Part |
|---|---|
| 0:00 - 0:15 | Part 1. The curse of dimensionality |
| 0:15 - 0:40 | Part 2. Kernel smoothing |
| 0:40 - 1:05 | Part 3. Splines |
| 1:05 - 1:20 | Part 4. Linear smoothers and effective degrees of freedom |
| 1:20 - 1:40 | Part 5. The kernel trick |
| 1:40 - 2:00 | Part 6. Gaussian processes, and a nonlinear Phillips curve |

## Setup

In [ ]:
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/korobilis/ECON5129-labs/main"

if not os.path.exists("econ5129_utils.py"):
    urllib.request.urlretrieve(f"{REPO_RAW}/econ5129_utils.py", "econ5129_utils.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import econ5129_utils as e5

e5.set_style()
rng = np.random.default_rng(5129)

## Part 1. The curse of dimensionality

Every local method rests on the same premise: to estimate $f(\mathbf{x}_0)$, average the responses of observations near $\mathbf{x}_0$. In high dimensions there are no observations near $\mathbf{x}_0$.

Draw $n$ points uniformly in the unit cube $[0,1]^d$ and measure the distance from the centre to the closest one.

In [ ]:
n_points = 500
dims = [1, 2, 3, 5, 10, 20, 50, 100]

median_distance = []
for d in dims:
    distances = []
    for _ in range(200):
        pts = rng.uniform(0, 1, size=(n_points, d))
        centre = np.full(d, 0.5)
        distances.append(np.min(np.sqrt(((pts - centre) ** 2).sum(axis=1))))
    median_distance.append(np.median(distances))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dims, median_distance, marker="o")
axes[0].set_xscale("log")
axes[0].set_xlabel("dimension $d$")
axes[0].set_ylabel("distance to nearest neighbour")
axes[0].set_title(f"Nearest neighbour of the centre, $n$ = {n_points}")

edge = np.array([0.01 ** (1.0 / d) for d in dims])
axes[1].plot(dims, edge, marker="s", color=e5.COLORS[2])
axes[1].set_xscale("log")
axes[1].set_xlabel("dimension $d$")
axes[1].set_ylabel("edge length of the neighbourhood")
axes[1].set_title("Cube edge needed to capture 1% of the data")
fig.tight_layout()
plt.show()

To capture even 1% of the sample in ten dimensions the neighbourhood must span roughly 63% of the range of every variable. It is no longer a neighbourhood, and any method that averages over it is no longer local. This is why the methods in this lab are used with a handful of predictors, and why Lecture 7 turns to a different strategy entirely.

## Part 2. Kernel smoothing

The Nadaraya-Watson estimator is a weighted average of the responses,

$$ \widehat{f}(x_0) = \frac{\sum_{i=1}^{n} K_h(x_0, x_i)\, y_i}{\sum_{i=1}^{n} K_h(x_0, x_i)}, \qquad K_h(x_0, x) = \exp\left\{-\frac{(x_0 - x)^2}{2h^2}\right\}, $$

with the bandwidth $h$ controlling how quickly the weights decay.

In [ ]:
def f_true(x):
    return np.sin(2 * np.pi * x) + 0.5 * x


n, noise = 120, 0.30
x = np.sort(rng.uniform(0, 1, n))
y = f_true(x) + noise * rng.normal(size=n)
grid = np.linspace(0, 1, 400)


def gaussian_kernel_weights(x_eval, x_data, h):
    """Matrix of Gaussian kernel weights, rows summing to one."""
    K = np.exp(-0.5 * ((x_eval[:, None] - x_data[None, :]) / h) ** 2)
    return K / K.sum(axis=1, keepdims=True)


def nadaraya_watson(x_eval, x_data, y_data, h):
    return gaussian_kernel_weights(x_eval, x_data, h) @ y_data


fig, ax = plt.subplots()
ax.scatter(x, y, s=18, color=e5.COLORS[6], alpha=0.7, label="data")
ax.plot(grid, f_true(grid), color=e5.COLORS[0], label="truth")
for h, colour in zip([0.01, 0.05, 0.3], [e5.COLORS[2], e5.COLORS[4], e5.COLORS[3]]):
    ax.plot(grid, nadaraya_watson(grid, x, y, h), color=colour, label=f"$h$ = {h}")
ax.set_title("Bandwidth controls the bias-variance trade-off")
ax.legend(fontsize=9)
plt.show()

### Boundary bias

Near the edges of the sample the kernel window is one-sided, so the weighted average is pulled towards the interior wherever the function has a slope. Local linear regression fixes this by fitting a line rather than a constant in each window,

$$ \min_{a, b} \sum_{i=1}^{n} K_h(x_0, x_i)\left[ y_i - a - b(x_i - x_0) \right]^2, $$

and reporting $\widehat{a}$.

In [ ]:
def local_linear(x_eval, x_data, y_data, h):
    """Local linear regression at each evaluation point."""
    fitted = np.empty(len(x_eval))
    for j, x0 in enumerate(x_eval):
        w = np.exp(-0.5 * ((x_data - x0) / h) ** 2)
        Z = np.column_stack([np.ones(len(x_data)), x_data - x0])
        WZ = Z * w[:, None]
        coef = np.linalg.solve(Z.T @ WZ + 1e-10 * np.eye(2), WZ.T @ y_data)
        fitted[j] = coef[0]
    return fitted


h = 0.05
fig, ax = plt.subplots()
ax.scatter(x, y, s=16, color=e5.COLORS[6], alpha=0.6)
ax.plot(grid, f_true(grid), color=e5.COLORS[0], label="truth")
ax.plot(grid, nadaraya_watson(grid, x, y, h), color=e5.COLORS[2], label="Nadaraya-Watson")
ax.plot(grid, local_linear(grid, x, y, h), color=e5.COLORS[4], label="local linear")
ax.set_xlim(-0.02, 0.25)
ax.set_title("Behaviour at the left boundary")
ax.legend(fontsize=9)
plt.show()

### Exercise 1

Choose the bandwidth by cross-validation rather than by eye.

1. Write `loocv_score(h)` returning the leave-one-out cross-validation error of the Nadaraya-Watson estimator. For a linear smoother there is a shortcut: the leave-one-out residual equals $\widehat{\varepsilon}_i / (1 - S_{ii})$, so no refitting is needed.
2. Evaluate it over a grid of bandwidths from 0.005 to 0.5 and plot the curve.
3. Compare the selected bandwidth with the value that minimises error against the true function, which you can compute because you simulated the data.

In [ ]:
def loocv_score(h):  #@keep
    """Leave-one-out cross-validation error using the smoother matrix diagonal."""  #@keep
    S = gaussian_kernel_weights(x, x, h)
    fitted = S @ y
    return np.mean(((y - fitted) / (1.0 - np.diag(S))) ** 2)


h_grid = np.logspace(np.log10(0.005), np.log10(0.5), 60)
cv_curve = np.array([loocv_score(h) for h in h_grid])
true_curve = np.array([e5.mse(f_true(grid), nadaraya_watson(grid, x, y, h)) for h in h_grid])

h_cv = h_grid[int(np.argmin(cv_curve))]
h_oracle = h_grid[int(np.argmin(true_curve))]

fig, ax = plt.subplots()
ax.plot(h_grid, cv_curve, label="leave-one-out CV")
ax.plot(h_grid, true_curve, label="error against the true function")
ax.axvline(h_cv, color=e5.COLORS[2], ls="--", label=f"CV choice: $h$ = {h_cv:.3f}")
ax.axvline(h_oracle, color=e5.COLORS[4], ls=":", label=f"oracle: $h$ = {h_oracle:.3f}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("bandwidth $h$")
ax.set_ylabel("mean squared error")
ax.legend(fontsize=9)
plt.show()

## Part 3. Splines

A regression spline is a piecewise polynomial constrained to join smoothly at a set of knots. The truncated power basis for a cubic spline with knots $\xi_1, \dots, \xi_K$ is

$$ \left\{1, x, x^2, x^3, (x - \xi_1)_+^3, \dots, (x - \xi_K)_+^3\right\}, $$

where $(u)_+ = \max(u, 0)$. Once the basis is built, fitting is ordinary least squares, exactly as in Lab 1.

In [ ]:
def spline_basis(x_values, knots):
    """Cubic truncated power basis."""
    powers = np.column_stack([x_values ** k for k in range(4)])
    truncated = np.column_stack([np.maximum(x_values - xi, 0.0) ** 3 for xi in knots])
    return np.column_stack([powers, truncated])


fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, n_knots in zip(axes, [2, 6, 25]):
    knots = np.quantile(x, np.linspace(0, 1, n_knots + 2)[1:-1])
    B = spline_basis(x, knots)
    coef, *_ = np.linalg.lstsq(B, y, rcond=None)

    ax.scatter(x, y, s=14, color=e5.COLORS[6], alpha=0.6)
    ax.plot(grid, f_true(grid), color=e5.COLORS[0])
    ax.plot(grid, spline_basis(grid, knots) @ coef, color=e5.COLORS[2])
    for xi in knots:
        ax.axvline(xi, color=e5.COLORS[6], lw=0.5, alpha=0.5)
    ax.set_title(f"{n_knots} knots")
fig.suptitle("Regression splines: more knots, more flexibility")
fig.tight_layout()
plt.show()

Choosing the number and position of knots is awkward. The penalised alternative places a knot at every observation and controls smoothness with a single penalty on the wiggliness of the fit, which is far easier to tune.

In [ ]:
def penalised_spline(x_eval, x_data, y_data, lam, n_knots=30):
    """Cubic spline with a ridge penalty on the truncated basis terms."""
    knots = np.quantile(x_data, np.linspace(0, 1, n_knots + 2)[1:-1])
    B = spline_basis(x_data, knots)
    penalty = np.diag([0.0] * 4 + [1.0] * len(knots))
    coef = np.linalg.solve(B.T @ B + lam * penalty, B.T @ y_data)
    return spline_basis(x_eval, knots) @ coef, coef, knots


fig, ax = plt.subplots()
ax.scatter(x, y, s=16, color=e5.COLORS[6], alpha=0.6)
ax.plot(grid, f_true(grid), color=e5.COLORS[0], label="truth")
for lam, colour in zip([1e-6, 1e-2, 1e2], [e5.COLORS[2], e5.COLORS[4], e5.COLORS[3]]):
    fit, _, _ = penalised_spline(grid, x, y, lam)
    ax.plot(grid, fit, color=colour, label=f"$\\lambda$ = {lam:g}")
ax.set_title("One penalty replaces the knot selection problem")
ax.legend(fontsize=9)
plt.show()

## Part 4. Linear smoothers and effective degrees of freedom

Every estimator so far shares a structure:

$$ \widehat{\mathbf{y}} = \mathbf{S}\mathbf{y}, $$

where $\mathbf{S}$ depends on the predictors but not on $\mathbf{y}$. Least squares, $k$-nearest neighbours, kernel smoothers and splines are all linear smoothers, and their complexity can be compared on a single scale: the **effective degrees of freedom** $\operatorname{tr}(\mathbf{S})$.

For least squares with $p$ regressors this equals $p$ exactly. For everything else it is a real number that moves continuously with the tuning parameter.

In [ ]:
def smoother_matrix_spline(x_data, lam, n_knots=30):
    knots = np.quantile(x_data, np.linspace(0, 1, n_knots + 2)[1:-1])
    B = spline_basis(x_data, knots)
    penalty = np.diag([0.0] * 4 + [1.0] * len(knots))
    return B @ np.linalg.solve(B.T @ B + lam * penalty, B.T)


def smoother_matrix_knn(x_data, k):
    n = len(x_data)
    S = np.zeros((n, n))
    order = np.argsort(np.abs(x_data[:, None] - x_data[None, :]), axis=1)[:, :k]
    for i in range(n):
        S[i, order[i]] = 1.0 / k
    return S


rows = []
for h_val in [0.02, 0.05, 0.15]:
    rows.append({"smoother": f"Nadaraya-Watson, $h$ = {h_val}",
                 "effective df": np.trace(gaussian_kernel_weights(x, x, h_val))})
for lam_val in [1e-6, 1e-3, 1e0, 1e3]:
    rows.append({"smoother": f"penalised spline, $\\lambda$ = {lam_val:g}",
                 "effective df": np.trace(smoother_matrix_spline(x, lam_val))})
for k_val in [3, 10, 40]:
    rows.append({"smoother": f"$k$-nearest neighbours, $k$ = {k_val}",
                 "effective df": np.trace(smoother_matrix_knn(x, k_val))})

print(pd.DataFrame(rows).set_index("smoother").round(2))

### Exercise 2

Effective degrees of freedom makes different families comparable. Use it to run a fair contest.

1. For each of the three smoothers, find the tuning parameter that delivers approximately 8 effective degrees of freedom.
2. Plot the three fits on one figure together with the truth.
3. Report the mean squared error of each against the true function on the grid.

At matched complexity, does the choice of smoother matter much?

In [ ]:
from scipy import optimize  #@keep

target_df = 8.0  #@keep

h_star = optimize.brentq(lambda hh: np.trace(gaussian_kernel_weights(x, x, hh)) - target_df,
                         0.005, 0.5)
lam_star = optimize.brentq(
    lambda ll: np.trace(smoother_matrix_spline(x, np.exp(ll))) - target_df, -15, 15)
lam_star = np.exp(lam_star)
k_star = min(range(2, 60), key=lambda kk: abs(np.trace(smoother_matrix_knn(x, kk)) - target_df))

spline_fit, _, _ = penalised_spline(grid, x, y, lam_star)
knn_fit = np.array([np.mean(y[np.argsort(np.abs(x - g))[:k_star]]) for g in grid])
nw_fit = nadaraya_watson(grid, x, y, h_star)

fig, ax = plt.subplots()
ax.scatter(x, y, s=14, color=e5.COLORS[6], alpha=0.5)
ax.plot(grid, f_true(grid), color=e5.COLORS[0], lw=2, label="truth")
ax.plot(grid, nw_fit, color=e5.COLORS[2], label=f"Nadaraya-Watson, $h$ = {h_star:.3f}")
ax.plot(grid, spline_fit, color=e5.COLORS[4], label=f"spline, $\\lambda$ = {lam_star:.2g}")
ax.plot(grid, knn_fit, color=e5.COLORS[3], label=f"$k$-NN, $k$ = {k_star}")
ax.set_title(f"Three smoothers at {target_df:.0f} effective degrees of freedom")
ax.legend(fontsize=9)
plt.show()

print(pd.DataFrame({
    "smoother": ["Nadaraya-Watson", "penalised spline", "k-nearest neighbours"],
    "MSE against truth": [e5.mse(f_true(grid), nw_fit),
                          e5.mse(f_true(grid), spline_fit),
                          e5.mse(f_true(grid), knn_fit)],
}).set_index("smoother").round(5))

## Part 5. The kernel trick

Basis expansions run into trouble when the basis is large: with $p$ predictors and degree $d$ polynomials there are $O(p^d)$ terms. The kernel trick sidesteps this. If the fitted values depend on the data only through inner products, we can replace every inner product with a kernel evaluation and work in the implied feature space without ever constructing it.

Kernel ridge regression is the cleanest example. The solution is

$$ \widehat{f}(x_0) = \mathbf{k}(x_0)' (\mathbf{K} + \lambda \mathbf{I})^{-1} \mathbf{y}, $$

where $\mathbf{K}$ collects kernel evaluations between training points and $\mathbf{k}(x_0)$ between $x_0$ and the training points. The Gaussian kernel corresponds to an infinite-dimensional basis.

In [ ]:
def rbf_kernel(A, B, lengthscale=0.1, amplitude=1.0):
    """Squared exponential kernel matrix."""
    sq = (A[:, None] - B[None, :]) ** 2
    return amplitude ** 2 * np.exp(-0.5 * sq / lengthscale ** 2)


def kernel_ridge(x_eval, x_data, y_data, lengthscale=0.1, lam=1e-2):
    K = rbf_kernel(x_data, x_data, lengthscale)
    alpha = np.linalg.solve(K + lam * np.eye(len(y_data)), y_data)
    return rbf_kernel(x_eval, x_data, lengthscale) @ alpha


fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ls in [0.02, 0.1, 0.4]:
    axes[0].plot(grid, kernel_ridge(grid, x, y, lengthscale=ls), label=f"lengthscale = {ls}")
axes[0].scatter(x, y, s=12, color=e5.COLORS[6], alpha=0.5)
axes[0].plot(grid, f_true(grid), color="black", lw=1.2, ls="--")
axes[0].set_title("Varying the lengthscale, $\\lambda$ = 0.01")
axes[0].legend(fontsize=9)

for lam_k in [1e-6, 1e-2, 1.0]:
    axes[1].plot(grid, kernel_ridge(grid, x, y, lengthscale=0.1, lam=lam_k),
                 label=f"$\\lambda$ = {lam_k:g}")
axes[1].scatter(x, y, s=12, color=e5.COLORS[6], alpha=0.5)
axes[1].plot(grid, f_true(grid), color="black", lw=1.2, ls="--")
axes[1].set_title("Varying the penalty, lengthscale = 0.1")
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

## Part 6. Gaussian processes

A Gaussian process places a prior directly on the function. Any finite collection of function values is jointly Gaussian with covariance given by the kernel,

$$ f \sim \mathcal{GP}(0, k(\cdot, \cdot)). $$

Conditioning on observed data gives a posterior that is also Gaussian, with mean

$$ \mathbb{E}[f(x_0) \mid \mathbf{y}] = \mathbf{k}(x_0)'(\mathbf{K} + \sigma^2\mathbf{I})^{-1}\mathbf{y} $$

and variance

$$ \operatorname{Var}[f(x_0) \mid \mathbf{y}] = k(x_0, x_0) - \mathbf{k}(x_0)'(\mathbf{K} + \sigma^2\mathbf{I})^{-1}\mathbf{k}(x_0). $$

The mean is exactly kernel ridge regression. What the Gaussian process adds is the variance.

In [ ]:
def gp_posterior(x_eval, x_data, y_data, lengthscale=0.1, amplitude=1.0, sigma2=0.09):
    """Posterior mean and standard deviation of a zero-mean Gaussian process."""
    K = rbf_kernel(x_data, x_data, lengthscale, amplitude) + sigma2 * np.eye(len(y_data))
    Ks = rbf_kernel(x_eval, x_data, lengthscale, amplitude)
    Kss = rbf_kernel(x_eval, x_eval, lengthscale, amplitude)

    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_data))
    mean = Ks @ alpha

    v = np.linalg.solve(L, Ks.T)
    cov = Kss - v.T @ v
    return mean, np.sqrt(np.maximum(np.diag(cov), 0.0)), cov


mean, sd, cov = gp_posterior(grid, x, y, lengthscale=0.1, amplitude=1.0, sigma2=noise ** 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

prior_draws = np.linalg.cholesky(
    rbf_kernel(grid, grid, 0.1) + 1e-8 * np.eye(len(grid))) @ rng.normal(size=(len(grid), 5))
axes[0].plot(grid, prior_draws, lw=1.0)
axes[0].set_title("Five draws from the prior")

posterior_draws = np.linalg.cholesky(
    cov + 1e-8 * np.eye(len(grid))) @ rng.normal(size=(len(grid), 5)) + mean[:, None]
axes[1].fill_between(grid, mean - 1.96 * sd, mean + 1.96 * sd, color=e5.COLORS[1], alpha=0.25)
axes[1].plot(grid, posterior_draws, lw=0.8, alpha=0.7)
axes[1].plot(grid, mean, color=e5.COLORS[0], lw=2, label="posterior mean")
axes[1].plot(grid, f_true(grid), color=e5.COLORS[2], ls="--", label="truth")
axes[1].scatter(x, y, s=12, color=e5.COLORS[6], alpha=0.6)
axes[1].set_title("Posterior after seeing the data")
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

### Exercise 3

The hyperparameters are not free choices: the marginal likelihood scores them. For a zero-mean Gaussian process,

$$ \log p(\mathbf{y}) = -\frac{1}{2}\mathbf{y}'(\mathbf{K} + \sigma^2\mathbf{I})^{-1}\mathbf{y} - \frac{1}{2}\log\left|\mathbf{K} + \sigma^2\mathbf{I}\right| - \frac{n}{2}\log(2\pi). $$

1. Write a function returning the negative log marginal likelihood as a function of the lengthscale, the amplitude and the noise variance, all parametrised in logs.
2. Minimise it with `scipy.optimize.minimize`.
3. Plot the resulting fit and compare the estimated noise variance with the value used to simulate the data.

In [ ]:
def neg_log_marginal(theta, x_data, y_data):  #@keep
    """Negative log marginal likelihood; theta = log(lengthscale, amplitude, sigma)."""  #@keep
    ls, amp, sig = np.exp(theta)
    K = rbf_kernel(x_data, x_data, ls, amp) + sig ** 2 * np.eye(len(y_data))
    try:
        L = np.linalg.cholesky(K)
    except np.linalg.LinAlgError:
        return 1e10
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_data))
    return float(0.5 * y_data @ alpha + np.sum(np.log(np.diag(L)))
                 + 0.5 * len(y_data) * np.log(2 * np.pi))


opt = optimize.minimize(neg_log_marginal, np.log([0.2, 1.0, 0.3]),
                        args=(x, y), method="Nelder-Mead",
                        options={"maxiter": 2000, "xatol": 1e-6, "fatol": 1e-6})
ls_hat, amp_hat, sig_hat = np.exp(opt.x)

print(f"lengthscale : {ls_hat:.4f}")
print(f"amplitude   : {amp_hat:.4f}")
print(f"noise sd    : {sig_hat:.4f}   (true value {noise})")

mean_opt, sd_opt, _ = gp_posterior(grid, x, y, ls_hat, amp_hat, sig_hat ** 2)

fig, ax = plt.subplots()
ax.fill_between(grid, mean_opt - 1.96 * sd_opt, mean_opt + 1.96 * sd_opt,
                color=e5.COLORS[1], alpha=0.25, label="95% band")
ax.plot(grid, mean_opt, color=e5.COLORS[0], label="posterior mean")
ax.plot(grid, f_true(grid), color=e5.COLORS[2], ls="--", label="truth")
ax.scatter(x, y, s=14, color=e5.COLORS[6], alpha=0.6)
ax.set_title("Gaussian process with hyperparameters estimated by marginal likelihood")
ax.legend(fontsize=9)
plt.show()

## Part 7. A nonlinear Phillips curve

Whether the relationship between unemployment and inflation is nonlinear is an old question with a large literature and direct policy consequences. It is also a two-dimensional problem, so the methods in this lab apply directly.

The response is annualised monthly CPI inflation; the predictor is the level of the unemployment rate.

In [ ]:
data, codes = e5.load_fredmd()
sample = data.loc["1960-01":"2019-12", ["CPIAUCSL", "UNRATE"]].dropna()

inflation = 1200 * np.log(sample["CPIAUCSL"]).diff()
frame = pd.DataFrame({"inflation": inflation, "unemployment": sample["UNRATE"]}).dropna()

u = frame["unemployment"].to_numpy()
pi = frame["inflation"].to_numpy()
u_grid = np.linspace(u.min(), u.max(), 200)

fig, ax = plt.subplots()
ax.scatter(u, pi, s=10, color=e5.COLORS[6], alpha=0.4, label="monthly observations")

b_lin, *_ = np.linalg.lstsq(np.column_stack([np.ones(len(u)), u]), pi, rcond=None)
ax.plot(u_grid, b_lin[0] + b_lin[1] * u_grid, color=e5.COLORS[0], label="linear")
ax.plot(u_grid, local_linear(u_grid, u, pi, h=0.6), color=e5.COLORS[2], label="local linear")

mean_pc, sd_pc, _ = gp_posterior(u_grid, u, pi - pi.mean(), lengthscale=1.0,
                                 amplitude=np.std(pi), sigma2=np.var(pi))
ax.plot(u_grid, mean_pc + pi.mean(), color=e5.COLORS[4], label="Gaussian process")
ax.fill_between(u_grid, mean_pc + pi.mean() - 1.96 * sd_pc, mean_pc + pi.mean() + 1.96 * sd_pc,
                color=e5.COLORS[4], alpha=0.15)

ax.set_xlabel("unemployment rate, %")
ax.set_ylabel("annualised monthly CPI inflation, %")
ax.set_title("The Phillips curve, fitted three ways")
ax.legend(fontsize=9)
plt.show()

### Exercise 4

Eyeballing a curve is not evidence. Test whether the nonlinearity earns its keep.

1. Split the sample chronologically at 1995, training on the earlier period.
2. Fit the linear model, a penalised spline and kernel ridge regression on the training period, choosing tuning parameters by cross-validation within it.
3. Compare out-of-sample mean squared error on the later period.

Does allowing for nonlinearity improve forecasts of inflation, or only improve the fit?

In [ ]:
train_mask = frame.index < "1995-01-01"  #@keep
u_tr, pi_tr = u[train_mask], pi[train_mask]  #@keep
u_te, pi_te = u[~train_mask], pi[~train_mask]  #@keep

b_tr, *_ = np.linalg.lstsq(np.column_stack([np.ones(len(u_tr)), u_tr]), pi_tr, rcond=None)
pred_linear = b_tr[0] + b_tr[1] * u_te

lam_candidates = np.logspace(-4, 6, 40)
cv_spline = []
for lam_c in lam_candidates:
    S = smoother_matrix_spline(u_tr, lam_c)
    fitted = S @ pi_tr
    cv_spline.append(np.mean(((pi_tr - fitted) / (1.0 - np.diag(S))) ** 2))
lam_best = lam_candidates[int(np.argmin(cv_spline))]
pred_spline, _, _ = penalised_spline(u_te, u_tr, pi_tr, lam_best)

pred_kernel = kernel_ridge(u_te, u_tr, pi_tr - pi_tr.mean(), lengthscale=1.0, lam=1.0) + pi_tr.mean()

print(pd.DataFrame({
    "model": ["linear", f"penalised spline ($\\lambda$ = {lam_best:.3g})", "kernel ridge"],
    "out-of-sample MSE": [e5.mse(pi_te, pred_linear),
                          e5.mse(pi_te, pred_spline),
                          e5.mse(pi_te, pred_kernel)],
}).set_index("model").round(4))

## Take-home challenges

1. **Additive models.** With two predictors, unemployment and lagged inflation, fit an additive model $f_1(u_t) + f_2(\pi_{t-1})$ by backfitting: smooth the residuals from one component against the other predictor, and iterate. Compare with a fully bivariate smoother and comment on the curse of dimensionality.

2. **Kernel choice.** Repeat Part 6 with a Matérn kernel with $\nu = 3/2$, whose sample paths are less smooth than the squared exponential. Which fits the inflation data better according to marginal likelihood?

3. **Confidence bands and honesty.** For the penalised spline, compute pointwise standard errors from $\sigma^2 \mathbf{S}\mathbf{S}'$ and compare the resulting bands with the Gaussian process posterior bands. Where do they disagree, and which would you trust at the edges of the data?

4. **Time variation.** Estimate the Phillips curve separately on 1960-1984 and 1985-2019 using the same smoother. Has the curve flattened, and can you distinguish flattening from a change in the range of unemployment observed?

---

**Next**: Lecture 6 keeps the nonlinearity but abandons smoothness. Lab 6 builds regression trees, then bags, randomises and boosts them.